In [1]:
# Importing useful dependencies
import torch
import boto3
import random
import functools
import numpy as np
from PIL import Image
from io import BytesIO
from typing import List, Tuple
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import CLIPProcessor, CLIPModel, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from torch import nn
# Set a seed for reproducibility
def set_seed(SEED=42):
  random.seed(SEED)
  np.random.seed(SEED)
  torch.manual_seed(SEED)
  torch.cuda.manual_seed_all(SEED)

set_seed(10721)

In [2]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url="http://127.0.0.1:9000", # MinIO API endpoint
    aws_access_key_id="minioadmin", # User name
    aws_secret_access_key="minioadmin", # Password
)

In [3]:
# Just in case our device has gpu
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load CLIP ViT-L/16

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch16",
    dtype=torch.float32,
    device_map="cuda"
)
# Load the processor (handles tokenization + image preprocessing)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
model.eval()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [4]:
# LoRA configuration TEXT
lora_config_text = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION, # for the text encoder
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none"
)
# LoRA configuration IMAGE
lora_config_image = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION, # for the image encoder
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none"
)
# --- Safe wrapper to filter allowed arguments ---
class QLoRACLIPWrapper(nn.Module):
    def __init__(self, model, model_type="vision"):
        super().__init__()
        self.model = model
        self.model_type = model_type

    def forward(self, **kwargs):
        if self.model_type == "vision":
            # call CLIPVisionTransformer normally and return the BaseModelOutput
            pixel_values = kwargs.get("pixel_values")
            return self.model.forward(pixel_values=pixel_values)
        elif self.model_type == "text":
            input_ids = kwargs.get("input_ids")
            attention_mask = kwargs.get("attention_mask", None)
            return self.model.forward(input_ids=input_ids, attention_mask=attention_mask)
        else:
            raise ValueError("model_type must be 'vision' or 'text'")

# Wrap text encoder
model.text_model = get_peft_model(QLoRACLIPWrapper(model.text_model, model_type="text"), lora_config_text)
# Wrap image encoder
model.vision_model = get_peft_model(QLoRACLIPWrapper(model.vision_model, model_type="vision"), lora_config_image)

In [5]:
# Create a model for all cases
model_0 = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch16",
    dtype=torch.float32,
    device_map="cuda"
)

model_0.text_model = get_peft_model(QLoRACLIPWrapper(model_0.text_model, model_type="text"), lora_config_text) # Wrap text encoder
model_0.vision_model = get_peft_model(QLoRACLIPWrapper(model_0.vision_model, model_type="vision"), lora_config_image) # Wrap image encoder
state_dict_0 = torch.load("../models/fine-tuned_LoRA_0.pt", map_location="cuda")
missing, unexpected = model_0.load_state_dict(state_dict_0, strict=False)

print("missing:", len(missing))
print("unexpected:", len(unexpected))

model_1 = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch16",
    dtype=torch.float32,
    device_map="cuda"
)

model_1.text_model = get_peft_model(QLoRACLIPWrapper(model_1.text_model, model_type="text"), lora_config_text) # Wrap text encoder
model_1.vision_model = get_peft_model(QLoRACLIPWrapper(model_1.vision_model, model_type="vision"), lora_config_image) # Wrap image encoder
state_dict_1 = torch.load("../models/fine-tuned_LoRA_1.pt", map_location="cuda")
missing1, unexpected1 = model_1.load_state_dict(state_dict_1, strict=False)

print("missing:", len(missing1))
print("unexpected:", len(unexpected1))

model_2 = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch16",
    dtype=torch.float32,
    device_map="cuda"
)

model_2.text_model = get_peft_model(QLoRACLIPWrapper(model_2.text_model, model_type="text"), lora_config_text) # Wrap text encoder
model_2.vision_model = get_peft_model(QLoRACLIPWrapper(model_2.vision_model, model_type="vision"), lora_config_image) # Wrap image encoder
state_dict_2 = torch.load("../models/fine-tuned_LoRA_2.pt", map_location="cuda")
missing2, unexpected2 = model_2.load_state_dict(state_dict_2, strict=False)

print("missing:", len(missing2))
print("unexpected:", len(unexpected2))
model_3 = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch16",
    dtype=torch.float32,
    device_map="cuda"
)

model_3.text_model = get_peft_model(QLoRACLIPWrapper(model_3.text_model, model_type="text"), lora_config_text) # Wrap text encoder
model_3.vision_model = get_peft_model(QLoRACLIPWrapper(model_3.vision_model, model_type="vision"), lora_config_image) # Wrap image encoder
state_dict_3 = torch.load("../models/fine-tuned_LoRA_3.pt", map_location="cuda")
missing3, unexpected3 = model_3.load_state_dict(state_dict_3, strict=False)

print("missing:", len(missing3))
print("unexpected:", len(unexpected3))



missing: 0
unexpected: 0
missing: 0
unexpected: 0
missing: 0
unexpected: 0
missing: 0
unexpected: 0


In [6]:
# Customized Dataloader class for our project
class ImageTextDataset(Dataset):
    def __init__(self, data_bucket, data_prefix, s3):
        self.data_bucket = data_bucket
        self.data_prefix = data_prefix
        self.s3 = s3

        # Data keys
        self.image_keys, self.text_keys = self.__loadfromminio__(data_bucket, data_prefix)

    def __loadfromminio__(self, data_bucket, data_prefix):
        image_keys = []
        text_keys = []
        paginator = self.s3.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=data_bucket, Prefix=data_prefix):
            for obj in page.get("Contents", []):
                  key = obj["Key"]
                  if obj['Size'] == 0 and key.endswith("/"):
                      continue
                  if "image" in key.split("/")[1]: # We only need images to find their corresponding description in MinIO
                      image_keys.append(key)
                      text_key = data_prefix + key.split("/")[1].replace("image", "text").replace("png", "txt")
                      text_keys.append(text_key)

        # From lists to arrays
        image_keys = np.array(image_keys)
        text_keys = np.array(text_keys)

        return image_keys, text_keys

    def __len__(self):
        return len(self.image_keys)

    def __getfile__(self, data_bucket, key, filetype = "image"):
        resp = self.s3.get_object(Bucket=data_bucket, Key=key)
        body = resp["Body"].read()
        if filetype == "image":
            file = Image.open(BytesIO(body))
        else: # filetype = "text"
            file = body.decode("utf-8")
        return file

    def __getitem__(self, idx):
        image = self.__getfile__(self.data_bucket, self.image_keys[idx], filetype = "image")

        # Load text
        text = self.__getfile__(self.data_bucket, self.text_keys[idx], filetype = "text")

        return image, text

# Collate function that applies preprocess and tokenizer to the batch
def collate_fn(batch: List[Tuple[Image.Image, str]], processor: CLIPProcessor):
    """
    batch: list of (PIL.Image, text_str)
    processor: Hugging Face CLIPProcessor
    Returns:
        images: torch.Tensor [B, C, H, W]
        input_ids: torch.LongTensor [B, L]
        attention_mask: torch.LongTensor [B, L]
        raw_texts: list[str]
    """
    images_pil, raw_texts = zip(*batch)

    # Use processor to handle both images and text
    processed = processor(
        text=list(raw_texts),
        images=list(images_pil),
        return_tensors="pt",
        padding=True,
        truncation=True,   # ensure sequences do not exceed max length
        max_length=77      # CLIP default
    )

    return (
        processed['pixel_values'],   # tensor [B, C, H, W]
        processed['input_ids'],      # tensor [B, L]
        processed['attention_mask'], # tensor [B, L]
        list(raw_texts)              # original texts
    )

# Wrap for DataLoader
collate = functools.partial(collate_fn, processor=processor)

In [7]:
# Create customized dataset objects
baseline_dataset = ImageTextDataset(
    data_bucket = "test-zone",
    data_prefix = "test-data/",
    s3 = s3
)

In [8]:
test_data = DataLoader(baseline_dataset, batch_size=8, shuffle=False, collate_fn=collate, num_workers=0)


In [9]:
# Collect embeddings
@torch.no_grad()
def collect_embeddings(model, dataloader, device):
    model.eval()

    all_image_feats = []
    all_text_feats = []

    for images, input_ids, attention_mask, _ in dataloader:
        images = images.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        img_feats = model.get_image_features(images)
        txt_feats = model.get_text_features(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Normalize (CRITICAL for cosine similarity)
        img_feats = F.normalize(img_feats, dim=-1)
        txt_feats = F.normalize(txt_feats, dim=-1)

        all_image_feats.append(img_feats)
        all_text_feats.append(txt_feats)

    all_image_feats = torch.cat(all_image_feats, dim=0)
    all_text_feats = torch.cat(all_text_feats, dim=0)

    return all_image_feats, all_text_feats


# Compute similarity matrix
def compute_similarity(image_feats, text_feats):
    # cosine similarity via dot product (since normalized)
    return text_feats @ image_feats.T


# Compute Recall@K and MRR
def retrieval_metrics(similarity, ks=(1, 5)):
    num_queries = similarity.size(0)

    # Ground truth: text i ↔ image i
    gt = torch.arange(num_queries, device=similarity.device)

    # Sort images by similarity (descending)
    rankings = torch.argsort(similarity, dim=1, descending=True)

    recalls = {k: 0 for k in ks}
    mrr = 0.0

    for i in range(num_queries):
        rank = (rankings[i] == gt[i]).nonzero(as_tuple=True)[0].item() + 1

        for k in ks:
            if rank <= k:
                recalls[k] += 1

        mrr += 1.0 / rank

    for k in ks:
        recalls[k] /= num_queries

    mrr /= num_queries

    return recalls, mrr






In [10]:
# Baseline model vs model 0
image_feats_base_0, text_feats_base_0 = collect_embeddings(model, test_data, device)
similarity_base_0 = compute_similarity(image_feats_base_0, text_feats_base_0)
recalls_base_0, mrr_base_0 = retrieval_metrics(similarity_base_0, ks=(1, 5))

image_feats_0, text_feats_0 = collect_embeddings(model_0, test_data, device)
similarity_0 = compute_similarity(image_feats_0, text_feats_0)
recalls_0, mrr_0 = retrieval_metrics(similarity_0, ks=(1, 5))

# model 1

image_feats_1, text_feats_1 = collect_embeddings(model_1, test_data, device)
similarity_1 = compute_similarity(image_feats_1, text_feats_1)
recalls_1, mrr_1 = retrieval_metrics(similarity_1, ks=(1, 5))

#  model 2

image_feats_2, text_feats_2 = collect_embeddings(model_2, test_data, device)
similarity_2 = compute_similarity(image_feats_2, text_feats_2)
recalls_2, mrr_2 = retrieval_metrics(similarity_2, ks=(1, 5))

#  model 3
image_feats_3, text_feats_3 = collect_embeddings(model_3, test_data, device)
similarity_3 = compute_similarity(image_feats_3, text_feats_3)
recalls_3, mrr_3 = retrieval_metrics(similarity_3, ks=(1, 5))

In [11]:
def print_results(name, recalls_base, mrr_base, recalls_ft, mrr_ft):
    print(f"\n===== {name} =====")
    print("Baseline model")
    print(f"Recall@1: {recalls_base[1]:.4f}")
    print(f"Recall@5: {recalls_base[5]:.4f}")
    print(f"MRR:      {mrr_base:.4f}")

    print("\nFine-tuned model")
    print(f"Recall@1: {recalls_ft[1]:.4f}")
    print(f"Recall@5: {recalls_ft[5]:.4f}")
    print(f"MRR:      {mrr_ft:.4f}")

print_results("Model 0", recalls_base_0, mrr_base_0, recalls_0, mrr_0)
print_results("Model 1", recalls_base_0, mrr_base_0, recalls_1, mrr_1)
print_results("Model 2", recalls_base_0, mrr_base_0, recalls_2, mrr_2)
print_results("Model 3", recalls_base_0, mrr_base_0, recalls_3, mrr_3)


===== Model 0 =====
Baseline model
Recall@1: 0.5040
Recall@5: 0.6913
MRR:      0.5952

Fine-tuned model
Recall@1: 0.5040
Recall@5: 0.6913
MRR:      0.5948

===== Model 1 =====
Baseline model
Recall@1: 0.5040
Recall@5: 0.6913
MRR:      0.5952

Fine-tuned model
Recall@1: 0.5040
Recall@5: 0.6913
MRR:      0.5940

===== Model 2 =====
Baseline model
Recall@1: 0.5040
Recall@5: 0.6913
MRR:      0.5952

Fine-tuned model
Recall@1: 0.5013
Recall@5: 0.6966
MRR:      0.5925

===== Model 3 =====
Baseline model
Recall@1: 0.5040
Recall@5: 0.6913
MRR:      0.5952

Fine-tuned model
Recall@1: 0.5040
Recall@5: 0.6966
MRR:      0.5930


In [12]:
# Function that returns different metrics for
def clip_retrieval_metrics(
    img_feats,
    txt_feats,
    logit_scale,
    n_bins=15,
):
    """
    img_feats: (N, D) normalized image embeddings
    txt_feats: (N, D) normalized text embeddings
    logit_scale: scalar tensor (CLIP logit_scale.exp())
    """

    device = img_feats.device
    N = img_feats.size(0)

    # Similarity logits
    logits_i2t = logit_scale * img_feats @ txt_feats.t()
    logits_t2i = logits_i2t.t()

    labels = torch.arange(N, device=device)

    # ---------- Accuracy ----------
    acc_i2t = (logits_i2t.argmax(dim=1) == labels).float().mean()
    acc_t2i = (logits_t2i.argmax(dim=1) == labels).float().mean()

    # ---------- Precision / Recall / F1 (Top-1) ----------
    def prf(logits):
        preds = logits.argmax(dim=1)
        tp = (preds == labels).sum().float()
        fp = (preds != labels).sum().float()
        fn = fp  # symmetric for retrieval

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        return precision, recall, f1

    prec_i2t, rec_i2t, f1_i2t = prf(logits_i2t)
    prec_t2i, rec_t2i, f1_t2i = prf(logits_t2i)

    # ---------- Cross-Entropy (CLIP contrastive loss) ----------
    loss_i2t = F.cross_entropy(logits_i2t, labels)
    loss_t2i = F.cross_entropy(logits_t2i, labels)
    ce_loss = (loss_i2t + loss_t2i) / 2

    # ---------- Expected Calibration Error (ECE) ----------
    def compute_ece(logits):
        probs = F.softmax(logits, dim=1)
        confidences, predictions = probs.max(dim=1)
        accuracies = predictions.eq(labels)

        ece = torch.zeros(1, device=device)
        bins = torch.linspace(0, 1, n_bins + 1, device=device)

        for i in range(n_bins):
            mask = (confidences > bins[i]) & (confidences <= bins[i + 1])
            if mask.any():
                acc = accuracies[mask].float().mean()
                conf = confidences[mask].mean()
                ece += (mask.float().mean()) * torch.abs(acc - conf)
        return ece

    ece_i2t = compute_ece(logits_i2t)
    ece_t2i = compute_ece(logits_t2i)

    return {
        "accuracy": {
            "i2t": acc_i2t.item(),
            "t2i": acc_t2i.item(),
        },
        "precision": {
            "i2t": prec_i2t.item(),
            "t2i": prec_t2i.item(),
        },
        "recall": {
            "i2t": rec_i2t.item(),
            "t2i": rec_t2i.item(),
        },
        "f1": {
            "i2t": f1_i2t.item(),
            "t2i": f1_t2i.item(),
        },
        "cross_entropy": ce_loss.item(),
        "ece": {
            "i2t": ece_i2t.item(),
            "t2i": ece_t2i.item(),
        },
    }

In [13]:
# Baseline model vs model 0
metrics_base_0 = clip_retrieval_metrics(
    image_feats_base_0,
    text_feats_base_0,
    model.logit_scale.exp()
)

metrics_model_0 = clip_retrieval_metrics(
    image_feats_0,
    text_feats_0,
    model_0.logit_scale.exp()
)
# model 0
metrics_model_1 = clip_retrieval_metrics(
    image_feats_1,
    text_feats_1,
    model_1.logit_scale.exp()
)

#  model 2


metrics_model_2 = clip_retrieval_metrics(
    image_feats_2,
    text_feats_2,
    model_2.logit_scale.exp()
)

#  model 3

metrics_model_3 = clip_retrieval_metrics(
    image_feats_3,
    text_feats_3,
    model_3.logit_scale.exp()
)

In [14]:
def print_metrics(name, m):
    print(f"\n===== {name} =====")
    print(f"Accuracy  (I→T / T→I): {m['accuracy']['i2t']:.4f} / {m['accuracy']['t2i']:.4f}")
    print(f"Precision (I→T / T→I): {m['precision']['i2t']:.4f} / {m['precision']['t2i']:.4f}")
    print(f"Recall    (I→T / T→I): {m['recall']['i2t']:.4f} / {m['recall']['t2i']:.4f}")
    print(f"F1-score  (I→T / T→I): {m['f1']['i2t']:.4f} / {m['f1']['t2i']:.4f}")
    print(f"CE Loss: {m['cross_entropy']:.4f}")
    print(f"ECE      (I→T / T→I): {m['ece']['i2t']:.4f} / {m['ece']['t2i']:.4f}")

# Baseline model vs model 0
print_metrics("Baseline model", metrics_base_0)
print_metrics("Fine-tuned model 0", metrics_model_0)
# Baseline model vs model 1
print_metrics("Baseline model", metrics_base_0)
print_metrics("Fine-tuned model 1", metrics_model_1)

# Baseline model vs model 2
print_metrics("Baseline model", metrics_base_0)
print_metrics("Fine-tuned model 2", metrics_model_2)

# Baseline model vs model 3
print_metrics("Baseline model", metrics_base_0)
print_metrics("Fine-tuned model 3", metrics_model_3)


===== Baseline model =====
Accuracy  (I→T / T→I): 0.4591 / 0.5040
Precision (I→T / T→I): 0.4591 / 0.5040
Recall    (I→T / T→I): 0.4591 / 0.5040
F1-score  (I→T / T→I): 0.4591 / 0.5040
CE Loss: 2.7394
ECE      (I→T / T→I): 0.1593 / 0.1029

===== Fine-tuned model 0 =====
Accuracy  (I→T / T→I): 0.4617 / 0.5040
Precision (I→T / T→I): 0.4617 / 0.5040
Recall    (I→T / T→I): 0.4617 / 0.5040
F1-score  (I→T / T→I): 0.4617 / 0.5040
CE Loss: 5.4757
ECE      (I→T / T→I): 0.4570 / 0.4994

===== Baseline model =====
Accuracy  (I→T / T→I): 0.4591 / 0.5040
Precision (I→T / T→I): 0.4591 / 0.5040
Recall    (I→T / T→I): 0.4591 / 0.5040
F1-score  (I→T / T→I): 0.4591 / 0.5040
CE Loss: 2.7394
ECE      (I→T / T→I): 0.1593 / 0.1029

===== Fine-tuned model 1 =====
Accuracy  (I→T / T→I): 0.4617 / 0.5040
Precision (I→T / T→I): 0.4617 / 0.5040
Recall    (I→T / T→I): 0.4617 / 0.5040
F1-score  (I→T / T→I): 0.4617 / 0.5040
CE Loss: 5.4747
ECE      (I→T / T→I): 0.4570 / 0.4994

===== Baseline model =====
Accuracy  (I

# Performance Evaluation Report: CLIP LoRA Fine-Tuning

This report evaluates the effectiveness of LoRA (Low-Rank Adaptation) fine-tuning on a CLIP ViT-B/16 model. We compare the baseline model against four fine-tuned variants (Models 0-3) using rank $r=8$ and $\alpha=32$.

---

## 1. Retrieval Performance Summary
The primary goal of the fine-tuning was to improve image-text alignment for retrieval tasks.

| Model | Recall@1 | Recall@5 | MRR |
| :--- | :---: | :---: | :---: |
| **Baseline** | **0.5040** | 0.6913 | **0.5952** |
| Model 0 | 0.5040 | 0.6913 | 0.5948 |
| Model 1 | 0.5040 | 0.6913 | 0.5940 |
| Model 2 | 0.5013 | **0.6966** | 0.5925 |
| **Model 3** | **0.5040** | **0.6966** | 0.5930 |

* **Marginal Gains**: Models 2 and 3 showed a slight improvement in **Recall@5** (up to 0.6966 from 0.6913).
* **Stagnant Top-1 Accuracy**: Recall@1 and Mean Reciprocal Rank (MRR) remained largely unchanged or saw minor decreases across all models.

---

## 2. Training Stability and Calibration
While retrieval metrics showed slight improvements, classification-based metrics indicate significant model degradation.

| Metric | Baseline | Model 3 (Best Retrieval) | Trend |
| :--- | :--- | :--- | :--- |
| **Accuracy (I→T / T→I)** | 0.4591 / 0.5040 | **0.4670** / 0.5040 | Slight Improvement |
| **CE Loss** | **2.7394** | 5.4743 | ❌ Severe Increase |
| **ECE (I→T)** | **0.1593** | 0.4623 | ❌ Poor Calibration |
| **ECE (T→I)** | **0.1029** | 0.4994 | ❌ Poor Calibration |

* **Cross-Entropy (CE) Loss**: The loss value nearly doubled after fine-tuning (from 2.73 to 5.47), suggesting the model's internal confidence is no longer aligned with the ground truth.
* **Calibration Collapse**: The **Expected Calibration Error (ECE)** quadrupled. An ECE near 0.50 indicates the model is extremely over-confident in its predictions, which can lead to unreliable performance in production environments.

---

## 3. Conclusion
1. **Performance**: Model 3 is the top performer in terms of retrieval accuracy, particularly for Recall@5.
2. **Issue**: There is a clear "calibration collapse" where the model's probability outputs are no longer trustworthy despite slightly better rankings.